#Design the Silver layer

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "viktoriia_kalenichenko_bronze")
dbutils.widgets.text("bronze_table", "")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("silver_table", "")
dbutils.widgets.text("key_columns", "")

In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table_name = dbutils.widgets.get("bronze_table")

silver_schema = dbutils.widgets.get("silver_schema")
silver_table_name = dbutils.widgets.get("silver_table")

key_columns = [
    c.strip()
    for c in dbutils.widgets.get("key_columns").split(",")
]

bronze_table = f"{catalog}.{bronze_schema}.{bronze_table_name}"
silver_table = f"{catalog}.{silver_schema}.{silver_table_name}"

In [0]:
#read bronze
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_df = spark.table(bronze_table)

In [0]:
#dedup
silver_clean_df = (bronze_df.dropDuplicates(key_columns).withColumn("silver_load_timestamp", F.current_timestamp()))

In [0]:
merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in key_columns])

In [0]:
if spark.catalog.tableExists(silver_table):
    before_count = spark.table(silver_table).count()
    silver_delta = DeltaTable.forName(spark, silver_table)

    (
        silver_delta.alias("target")
        .merge(
            silver_clean_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    action = "Silver MERGE completed"
else:
    before_count = 0
    (
        silver_clean_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
    action = "Silver table created"
after_count = spark.table(silver_table).count()

print(action)
print("Rows before:", before_count)
print("Rows after:", after_count)
print("Rows changed/inserted:", after_count - before_count)

In [0]:
#check for duplicates 
duplicate_count = (
    spark.table(silver_table)
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate keys:", duplicate_count)  

In [0]:
if duplicate_count > 0:
    raise ValueError(
        f"Data quality check failed: {duplicate_count} duplicate key groups found"
    )

For the `environment`, we don't need to rewrite anything in the code at all - we simply change widgets:

bronze_table
environment

silver_table
environment

key_columns
AREA,YEAR

widgets for the `yield_and_climate`:

bronze_table
yield_and_climate

silver_table
yield_and_climate

key_columns
AREA,ITEM,YEAR